# F1 Pit Stop Prediction - XGBoost Model

**Kaggle Playground Series S6E5**

Second base model. Same 38 features and GroupKFold CV as the LGB baseline. We then blend LGB and XGB OOF predictions and check if the ensemble beats either model alone.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import TargetEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 110
SEED = 42

print(f'XGBoost version: {xgb.__version__}')

## 1. Load and Engineer Features

In [ ]:
train_raw = pd.read_csv('../data/train.csv')
test_raw  = pd.read_csv('../data/test.csv')
sub       = pd.read_csv('../data/sample_submission.csv')

sort_keys = ['Race', 'Year', 'Driver', 'LapNumber']
train_raw = train_raw.sort_values(sort_keys).reset_index(drop=True)
test_raw  = test_raw.sort_values(sort_keys).reset_index(drop=True)

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')

In [ ]:
def add_tier1_features(df, compound_stats=None, is_train=True):
    df = df.copy()
    if is_train:
        compound_stats = df[df['PitNextLap'] == 1].groupby('Compound')['TyreLife'].agg(
            compound_pit_median='median', compound_pit_std='std').reset_index()
    df = df.merge(compound_stats, on='Compound', how='left')
    df['TyreLife_vs_PitWindow'] = df['TyreLife'] - df['compound_pit_median']
    df['TyreLife_PitWindow_Z']  = df['TyreLife_vs_PitWindow'] / (df['compound_pit_std'] + 1)
    df['In_PitWindow']          = (df['TyreLife'] >= df['compound_pit_median'] * 0.75).astype(int)
    df['Deg_Rate']              = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    df['LapsRemaining']         = 1.0 - df['RaceProgress']
    df['Already_Pitted']        = (df['Stint'] > 1).astype(int)
    df['Early_Race']            = (df['RaceProgress'] < 0.12).astype(int)
    df['Late_Race']             = (df['RaceProgress'] > 0.92).astype(int)
    df['Mid_Race']              = ((df['RaceProgress'] >= 0.25) & (df['RaceProgress'] <= 0.75)).astype(int)
    df['Position_x_TyreLife']   = df['Position'] * df['TyreLife']
    df['TyreLife_norm']         = df['TyreLife'] / df['compound_pit_median']
    return df, compound_stats

def add_tier2_features(df):
    df = df.copy()
    group = ['Race', 'Year', 'Driver']
    df['LapTime_Roll3']     = df.groupby(group)['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['LapTime_Roll5']     = df.groupby(group)['LapTime (s)'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df['LapTime_Trend3']    = df['LapTime (s)'] - df['LapTime_Roll3']
    df['LapTime_Trend5']    = df['LapTime (s)'] - df['LapTime_Roll5']
    df['LapTime_vs_Median'] = df.groupby(group)['LapTime (s)'].transform(lambda x: x - x.median())
    df['Deg_Roll3']         = df.groupby(group)['Cumulative_Degradation'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['Deg_Trend']         = df['Cumulative_Degradation'] - df['Deg_Roll3']
    df['PitStop_Lag1']      = df.groupby(group)['PitStop'].shift(1).fillna(0)
    df['TyreLife_Lag1']     = df.groupby(group)['TyreLife'].shift(1).fillna(0)
    df['LapTime_Lag1']      = df.groupby(group)['LapTime (s)'].shift(1).fillna(df['LapTime (s)'])
    df['Laps_Since_Pit']    = df.groupby(group)['PitStop'].transform(
        lambda x: x.groupby((x != x.shift()).cumsum()).cumcount())
    df['PosCh_Roll3']       = df.groupby(group)['Position_Change'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    return df

def add_tier3_features(train_df, test_df):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}
    train_df['Compound_enc'] = train_df['Compound'].map(compound_map).fillna(1)
    test_df['Compound_enc']  = test_df['Compound'].map(compound_map).fillna(1)
    te = TargetEncoder(target_type='binary', smooth='auto', random_state=SEED)
    train_df['Driver_enc'] = te.fit_transform(train_df[['Driver']], train_df['PitNextLap']).ravel()
    test_df['Driver_enc']  = te.transform(test_df[['Driver']]).ravel()
    te2 = TargetEncoder(target_type='binary', smooth='auto', random_state=SEED)
    train_df['Race_enc'] = te2.fit_transform(train_df[['Race']], train_df['PitNextLap']).ravel()
    test_df['Race_enc']  = te2.transform(test_df[['Race']]).ravel()
    dc_rate = (train_df.groupby(['Driver', 'Compound'])['PitNextLap'].mean()
               .reset_index().rename(columns={'PitNextLap': 'Driver_Compound_PitRate'}))
    train_df = train_df.merge(dc_rate, on=['Driver', 'Compound'], how='left')
    test_df  = test_df.merge(dc_rate,  on=['Driver', 'Compound'], how='left')
    test_df['Driver_Compound_PitRate'] = test_df['Driver_Compound_PitRate'].fillna(
        train_df['Driver_Compound_PitRate'].mean())
    return train_df, test_df

train, compound_stats = add_tier1_features(train_raw, is_train=True)
test,  _              = add_tier1_features(test_raw, compound_stats=compound_stats, is_train=False)
train = add_tier2_features(train)
test  = add_tier2_features(test)
train, test = add_tier3_features(train, test)

raw_features   = ['LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)',
                  'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress',
                  'Position_Change', 'PitStop', 'Year']
tier1_features = ['TyreLife_vs_PitWindow', 'TyreLife_PitWindow_Z', 'In_PitWindow',
                  'Deg_Rate', 'LapsRemaining', 'Already_Pitted', 'Early_Race',
                  'Late_Race', 'Mid_Race', 'Position_x_TyreLife', 'TyreLife_norm']
tier2_features = ['LapTime_Roll3', 'LapTime_Roll5', 'LapTime_Trend3', 'LapTime_Trend5',
                  'LapTime_vs_Median', 'Deg_Roll3', 'Deg_Trend', 'PitStop_Lag1',
                  'TyreLife_Lag1', 'LapTime_Lag1', 'Laps_Since_Pit', 'PosCh_Roll3']
tier3_features = ['Compound_enc', 'Driver_enc', 'Race_enc', 'Driver_Compound_PitRate']
FEATURES = raw_features + tier1_features + tier2_features + tier3_features

X      = train[FEATURES]
y      = train['PitNextLap']
X_test = test[FEATURES]
groups = train['Race'] + '_' + train['Year'].astype(str)

print(f'Features: {len(FEATURES)}  |  Train: {X.shape}  |  Test: {X_test.shape}')

## 2. XGBoost with GroupKFold CV

In [ ]:
xgb_params = {
    'objective':        'binary:logistic',
    'eval_metric':      'auc',
    'n_estimators':     3000,
    'learning_rate':    0.05,
    'max_depth':        7,
    'min_child_weight': 50,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'scale_pos_weight': 4,
    'tree_method':      'hist',
    'random_state':     SEED,
    'verbosity':        0,
}

gkf            = GroupKFold(n_splits=5)
xgb_oof        = np.zeros(len(train))
xgb_test_preds = np.zeros(len(test))
xgb_fold_aucs  = []
xgb_models     = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=150,
        verbose=500
    )

    xgb_oof[val_idx]    = model.predict_proba(X_val)[:, 1]
    xgb_test_preds     += model.predict_proba(X_test)[:, 1] / 5
    xgb_models.append(model)

    fold_auc = roc_auc_score(y_val, xgb_oof[val_idx])
    xgb_fold_aucs.append(fold_auc)
    print(f'Fold {fold+1}  |  AUC: {fold_auc:.5f}  |  Trees: {model.best_iteration}')

xgb_oof_auc = roc_auc_score(y, xgb_oof)
print(f'\nXGB OOF AUC: {xgb_oof_auc:.5f}')
print(f'Folds:       {[round(a, 5) for a in xgb_fold_aucs]}')

## 3. Load LGB OOF and Blend

We re-run LGB quickly to get the OOF predictions for blending. Alternatively load from notebook 02 if you already have them.

In [ ]:
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'n_estimators': 3000,
    'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 50,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': 4,
    'verbose': -1, 'random_state': SEED,
}

lgb_oof        = np.zeros(len(train))
lgb_test_preds = np.zeros(len(test))
lgb_fold_aucs  = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)])

    lgb_oof[val_idx]    = model.predict_proba(X_val)[:, 1]
    lgb_test_preds     += model.predict_proba(X_test)[:, 1] / 5

    fold_auc = roc_auc_score(y_val, lgb_oof[val_idx])
    lgb_fold_aucs.append(fold_auc)
    print(f'Fold {fold+1}  |  AUC: {fold_auc:.5f}  |  Trees: {model.best_iteration_}')

lgb_oof_auc = roc_auc_score(y, lgb_oof)
print(f'\nLGB OOF AUC: {lgb_oof_auc:.5f}')

## 4. Blend and Find Best Weights

In [ ]:
# Grid search over blend weights
best_auc    = 0
best_weight = 0.5
results     = []

for w in np.arange(0.0, 1.01, 0.05):
    blended = w * lgb_oof + (1 - w) * xgb_oof
    auc     = roc_auc_score(y, blended)
    results.append({'lgb_weight': round(w, 2), 'xgb_weight': round(1-w, 2), 'auc': round(auc, 5)})
    if auc > best_auc:
        best_auc    = auc
        best_weight = w

results_df = pd.DataFrame(results)
print(f'LGB standalone OOF AUC:  {lgb_oof_auc:.5f}')
print(f'XGB standalone OOF AUC:  {xgb_oof_auc:.5f}')
print(f'Best blend OOF AUC:      {best_auc:.5f}  (LGB weight: {best_weight:.2f}, XGB weight: {1-best_weight:.2f})')
print()
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(results_df['lgb_weight'], results_df['auc'], marker='o', color='steelblue', linewidth=2)
ax.axvline(best_weight, color='tomato', linestyle='--', linewidth=1.5, label=f'Best weight: {best_weight:.2f}')
ax.axhline(lgb_oof_auc, color='steelblue', linestyle=':', linewidth=1, label=f'LGB alone: {lgb_oof_auc:.5f}')
ax.axhline(xgb_oof_auc, color='orange',    linestyle=':', linewidth=1, label=f'XGB alone: {xgb_oof_auc:.5f}')
ax.set_title('Blend OOF AUC vs LGB weight')
ax.set_xlabel('LGB weight (XGB weight = 1 - this)')
ax.set_ylabel('OOF AUC')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Save Blended Submission

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

blended_test = best_weight * lgb_test_preds + (1 - best_weight) * xgb_test_preds

sub = pd.read_csv('../data/sample_submission.csv')
sub['PitNextLap'] = 0.0
sub_sorted = sub.sort_values('id').reset_index(drop=True)
sub_sorted['PitNextLap'] = blended_test[test.sort_values('id').index]
sub_sorted.to_csv('../submissions/submission_lgb_xgb_blend.csv', index=False)

print(f'Submission saved: submission_lgb_xgb_blend.csv')
print(f'LGB OOF AUC:    {lgb_oof_auc:.5f}')
print(f'XGB OOF AUC:    {xgb_oof_auc:.5f}')
print(f'Blend OOF AUC:  {best_auc:.5f}  (LGB: {best_weight:.2f}, XGB: {1-best_weight:.2f})')

## 6. Score Log

| Submission | OOF AUC | LB AUC | Notes |
|---|---|---|---|
| submission_lgb_baseline.csv | 0.93169 | 0.94525 | LGB alone |
| submission_lgb_xgb_blend.csv | ? | ? | Fill in after submitting |